In [1]:
from recommender_system.data.load import load_data
from recommender_system.data.transform import transform_reviews, train_test_split

df = load_data('../data/australian_user_reviews.json.gz')
reviews = transform_reviews(df)
train, test = train_test_split(reviews, n_test=1)

train.head()

,user_id,posted,item_id,recommend
58236,--ace--,2014-01-24,440,True
32158,--ionex--,2015-08-15,105600,True
45374,-2SV-vuLB-Kg,2014-10-07,440,True
45375,-2SV-vuLB-Kg,2014-10-15,200510,True
45373,-2SV-vuLB-Kg,2014-10-19,302510,True


In [20]:
from recommender_system.utils import recall_at_k
from recommender_system.models.popular import recommend_popular


recommendations = recommend_popular(train, n=3)
recommendations.head(10)

,user_id,item_id
0,--ace--,730
1,--ace--,4000
2,--ace--,570
3,--ionex--,440
4,--ionex--,730
5,--ionex--,4000
6,-2SV-vuLB-Kg,4000
7,-2SV-vuLB-Kg,570
8,-2SV-vuLB-Kg,304930
9,-Mad-,440


In [15]:
recommendations.info()

<class 'pandas.DataFrame'>
RangeIndex: 32676 entries, 0 to 32675
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   user_id  32676 non-null  str  
 1   item_id  32676 non-null  str  
dtypes: str(2)
memory usage: 510.7 KB


## Recall@3
n_test=1: 0.1132 (Baseline popularity model)

In [16]:
recall = recall_at_k(recommendations, test, k=3)
print(recall)

0.11320235034887992


In [18]:
print("Train users:", train["user_id"].nunique())
print("Test users:", test["user_id"].nunique())
print("Train interactions:", len(train))
print("Test interactions:", len(test))
train.groupby("user_id").size().describe()

Train users: 10892
Test users: 10892
Train interactions: 28349
Test interactions: 10892


count    10892.000000
mean         2.602736
std          2.007973
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         19.000000
dtype: float64

## Collaborative filtering

In [21]:
from recommender_system.models.collaborative_filtering import get_item_similarity_matrix, recommend_item_cf

sims, item_to_idx = get_item_similarity_matrix(train)

recommendations = recommend_item_cf(train, sims, item_to_idx, n=3)

## Recall@3
n_test=1: 0.05206

In [22]:
recall = recall_at_k(recommendations, test, k=3)
print(recall)

0.05205655526992288


## Matrix Factorization

In [ ]:
from recommender_system.models.matrixfactorization import MatrixFactorization, recommend_mf
from recommender_system.models.collaborative_filtering import df2interact_mat

interactions, item_to_idx, user_to_idx = df2interact_mat(train, 'user_id', 'item_id', 'recommend')

model = MatrixFactorization(n_factors=25, random_state=42)
model.fit(interactions, n_epochs=10, learning_rate=0.01, regularization=0)

Epoch 1/10, RMSE: 1.0268
Epoch 2/10, RMSE: 1.0218
Epoch 3/10, RMSE: 1.0166
Epoch 4/10, RMSE: 1.0105
Epoch 5/10, RMSE: 1.0026
Epoch 6/10, RMSE: 0.9919
Epoch 7/10, RMSE: 0.9764
Epoch 8/10, RMSE: 0.9548
Epoch 9/10, RMSE: 0.9271
Epoch 10/10, RMSE: 0.8963


In [ ]:
recommendations = recommend_mf(model, test, train, user_to_idx, {idx: item for item, idx in item_to_idx.items()}, n=3)
recommendations.head(10)

,user_id,item_id,score
0,--ace--,730,0.590259
1,--ace--,304930,0.539621
2,--ace--,4000,0.485933
3,--ionex--,730,0.526006
4,--ionex--,252490,0.248273
5,--ionex--,550,0.228009
6,-2SV-vuLB-Kg,304930,0.358194
7,-2SV-vuLB-Kg,4000,0.278009
8,-2SV-vuLB-Kg,221100,0.228721
9,-Mad-,72850,0.196015


## Recall@3

In [87]:
from recommender_system.utils import recall_at_k
recall = recall_at_k(recommendations, test, k=3)
print(recall)

0.060686742563349244
